## 1 - Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## 2 - Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 306.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 298.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 317.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 279.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.13 requires numpy<2,>=1, but you have numpy 2.3.2 which is incompatible.
langchain 0.1.1 requires numpy<2,>=1, but you have numpy 2.3.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
xarray 2025.7.1

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.23.2 pandas==1.5.3 tiktoken==0.6.0 pymupdf==1.25.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==0.4.22 sentence-transformers==2.3.1 numpy==1.25.2 posthog==4.0.0 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
xarray 2025.7.1 requires numpy>=1.26, but you have numpy 1.25.2 which is incompatible.
xarray 2025.7.1 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
xarray 2025.7.1 requires pandas>=2.2, but you have pandas 1.5.3 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.2 which is incompatible.
arviz 0.22.0 requires numpy>=1.26.0, but you have numpy 1.25.2 which is incompatible.
arviz 0.22.0 requires pandas>=2.1.0, but you have pandas 1.5.3 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.25.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0,

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd
pd.set_option('max_colwidth', None)

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [4]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3 - Set Constant Variables

This section is dedicated to defining all constant variables, thereby establishing a centralized location within the notebook for convenient and consistent access.

In [5]:
# Set maximum number of tokens for the model output.
MAX_TOKENS = 1024

# Set the name of the model to be use in Llama_cpp
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

# File path for the PDF document to be processed
FILENAME = "/content/drive/MyDrive/University of Texas/Medical Assistant Project/data/medical_diagnosis_manual.pdf"

The project defines a set of medical questions that the LLM model is expected to address using various methodologies, including basic LLM inference, prompt engineering, and retrieval-augmented generation (RAG). These questions will be encapsulated as variables to facilitate their application across different LLM techniques.

In [6]:
# Set the medical questions into variables.
query_1 = """
What is the protocol for managing sepsis in a critical care unit?
"""
query_2 = """
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
"""
query_3 = """
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
"""
query_4 = """
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
"""
query_5 = """
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
"""


This system message is designed for prompt engineering and sets up instructions for the LLM. It gives the model a clear role—as a professional medical assistant—and lays out guidelines before any questions are asked. This setup helps the model produce deeper, more thoughtful responses, rather than just short or surface-level answers to medical queries.

In [7]:
# Define system prompt
system_prompt = """
You are a professional medical assistant. Your role is to provide clear, accurate, and empathetic responses to medical questions based on your internal knowledge.
Always prioritize: - Clinical accuracy and patient safety. - Clear communication.
When responding: - Use medically accepted terminology and explain it when needed. - Clarify when information may be outdated or incomplete. - Encourage users to consult licensed healthcare professionals for any serious or personal medical concerns.
Maintain a calm, professional, and supportive tone. Use structured formats (e.g., bullet points, tables) when helpful.
Avoid speculation, hallucination, or overconfidence in uncertain areas.
Your goal is to educate, support decision-making, and promote health literacy.
"""

This system message is meant to guide the model to rely only on the context it’s given, instead of tapping into its broader training data. It clearly spells out that if the model can’t find an answer within that context, it should simply respond with “I don’t know.” The goal is to keep things grounded, avoiding guesswork, made-up information, or sounding overly confident when there’s uncertainty.

In [8]:
# Define system message for RAG
qna_system_message = """
Strictly respond only using the information in the context.
Do not include or reference the context or the question in your response.
If the answer is not derivable, say "I don't know".
Do not use prior knowledge or training data.
Use medically accepted terminology and structured formats when helpful.
Avoid speculation, hallucination, or overconfidence.
"""

In [9]:
# Define user message template for RAG
qna_user_message_template = """
###CONTEXT_START
{context}
###CONTEXT_END
###QUESTION_START
{question}
###QUESTION_END
"""

Define the system message for the Groundedness and Relevance Evaluations. Also the template to be used for the prompt that will be passed to the LLM in order to evaluate the responses.

In [10]:
# Define system message for groundedness evaluation
groundedness_rater_system_message  = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [11]:
# Define system message for relevance evaluation
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [12]:
# Define the template for the evaluation
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

The Merck Manual contains a footer in all the pages. This is the text in the footer of the manual.

In [13]:
# Footer present in all the manual pages
footer = (
    "fernando88@gmail.com\n"
    "EPFAT4LO8B\n"
    "This file is meant for personal use by fernando88@gmail.com only.\n"
    "Sharing or publishing the contents in part or full is liable for legal action.\n"
)

## 4 - Define Functions

This section defines a set of functions designed to generate prompts, retrieve responses from the LLM model, and evaluate the model's outputs.

In [14]:
# Function that generates a response from a language model (LLM) based on the provided query.
def response(query, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    """
    Generates a response from a language model (LLM) based on the provided query.

    Parameters:
        query (str): The input prompt or question to be passed to the LLM.
        max_tokens (int, optional): The maximum number of tokens to generate in the response. Default is 128.
        temperature (float, optional): Controls the randomness of the output. Lower values yield more deterministic responses. Default is 0.
        top_p (float, optional): Implements nucleus sampling by selecting tokens from the top probability mass. Default is 0.95.
        top_k (int, optional): Limits the sampling pool to the top-k most probable tokens. Default is 50.

    Returns:
        str: The textual output generated by the LLM in response to the input query.
    """
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [15]:
# Function that creates a prompt using a system message and a user query
def llama_prompt_builder(system_prompt, user_query):
    """
    Constructs a formatted prompt for the LLaMA model by combining a system message and a user query.

    Parameters:
        system_prompt (str): The system-level instruction or context intended to guide the model's behavior.
        user_query (str): The user's input or question to be addressed by the model.

    Returns:
        str: A structured prompt string formatted according to the LLaMA model's expected input syntax.
    """
    prompt = f"""<s>[INST] <<SYS>>
    {system_prompt}
    <</SYS>>
    {user_query} [/INST]
    """
    return prompt

In [16]:
# Function that constructs a RAG-style prompt using a system message and user-provided context and question.
def build_rag_prompt(system_message: str, context: str, question: str) -> str:
    """
    Constructs a prompt formatted for Retrieval-Augmented Generation (RAG) by integrating a system message,
    retrieved context from external sources, and a user-defined question.

    Parameters:
        system_message (str): The system-level instruction that guides the model's behavior and response style.
        context (str): External information or retrieved content intended to support the model's answer.
        question (str): The user's query that the model is expected to address using the provided context.

    Returns:
        str: A structured prompt string formatted for RAG workflows, suitable for input to LLaMA-style models.
    """
    user_message = f"""
Consider the following ###Context and ###Question
###Context
{context}

###Question
{question}
""".strip()

    prompt = f"""<s>[INST]<<SYS>>
{system_message.strip()}
<</SYS>>
{user_message.strip()} [/INST]"""

    return prompt


In [17]:
# Function that generates a response using a Retrieval-Augmented Generation (RAG) approach by leveraging external context.
def generate_rag_response(user_input, k=3, max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    """
    Generates a response using a Retrieval-Augmented Generation (RAG) approach by leveraging external context
    retrieved from a document corpus and integrating it into a structured prompt for the LLM.

    Parameters:
        user_input (str): The user's query or question to be answered by the model.
        k (int, optional): The number of top relevant document chunks to retrieve. Default is 3.
        max_tokens (int, optional): The maximum number of tokens to generate in the model's response. Default is 512.
        temperature (float, optional): Controls randomness in generation; lower values yield more deterministic output. Default is 0.
        top_p (float, optional): Nucleus sampling parameter; selects tokens from the top cumulative probability mass. Default is 0.95.
        top_k (int, optional): Limits token selection to the top-k most probable tokens. Default is 50.

    Returns:
        str: The model-generated response based on the retrieved context and user query. If an error occurs during generation,
             an error message is returned instead.
    """
    global qna_system_message, qna_user_message_template

    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    prompt = build_rag_prompt(qna_system_message, context_for_query, user_input)
    # print(f"Prompt:\n{prompt}")

    try:
        response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

In [18]:
# Function that generates a response to user input along with groundedness and relevance ratings.
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    """
    Generates a response to user input along with groundedness and relevance ratings.

    Parameters:
        user_input (str): The user's question or input to respond to
        k (int, optional): Number of relevant document chunks to retrieve. Defaults to 3.
        max_tokens (int, optional): Maximum number of tokens in generated responses. Defaults to 128.
        temperature (float, optional): Controls randomness in generation (0 = deterministic). Defaults to 0.
        top_p (float, optional): Nucleus sampling probability threshold. Defaults to 0.95.
        top_k (int, optional): Top-k sampling value. Defaults to 50.

    Returns:
        tuple: A tuple containing three elements:
            - groundedness_rating (str): The LLM's rating of how well-grounded the answer is in the context
            - relevance_rating (str): The LLM's rating of how relevant the answer is to the question
            - answer (str): The generated answer to the user's input
    """
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text'], answer

In [19]:
# Function that removes footer text from all pages in a document.
def remove_footer(pages, footer_text):
    """
    This function iterates through each page in the input list and removes the specified
    footer text if it exists in the page content.

    Parameters:
        pages (list): A list of page objects where each object has a 'page_content' attribute.
        footer_text (str): The text string to identify and remove from each page's footer.

    Returns:
        list: The same list of page objects with the footer text removed from their content.
    """
    for page in pages:
        if footer_text in page.page_content:
            page.page_content = page.page_content.replace(footer_text, "")
    return pages

## 5 - Downloading and Loading the model

This section outlines the procedure for downloading and initializing the Large Language Model (LLM) using tools from Hugging Face and the llama_cpp interface. The model is retrieved via the hf_hub_download utility, the chosen model is TheBloke/Llama-2-13B-chat-GGUF. Once downloaded, the model is instantiated using the Llama class.

The context window is set to 4096 tokens, enabling the model to process longer input sequences and maintain coherence across extended prompts. This is  beneficial for the prompt engineering and RAG sections of this project. This context window will allow to use bigger prompts that include context.

The model will include offloading 38 layers to the GPU. Allowing parallel processing and helping the performance of the model.

The model will use a batch size of 512, this is the amount of tokens processed simultaneously.

These settings help to optimize the model responsiveness across diverse NLP tasks.

In [20]:
# From Hugging Face download a model.
model_path = hf_hub_download(
    repo_id= model_name_or_path,
    filename= model_basename
)

llama-2-13b-chat.Q5_K_M.gguf:   0%|          | 0.00/9.23G [00:00<?, ?B/s]

In [21]:
# Initialize the Large Language Model
llm = Llama(
    model_path=model_path,
    n_ctx=4096,                  # Context length (max sequence length) 4096
    n_gpu_layers=38,             # Offload ~40 layers to GPU
    n_batch=512,                 # Batch size for prompt processing
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## 6 - Question Answering using LLM

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [22]:
response_llm_1 = response(query_1, max_tokens=MAX_TOKENS)
print(response_llm_1)


Sepsis is a life-threatening condition that can arise from an infection, and it is important to have a clear protocol for managing it in a critical care unit. Here are some key components of a sepsis management protocol:

1. Early recognition and identification: The first step in managing sepsis is to recognize the signs and symptoms early on, such as fever, tachycardia, tachypnea, and confusion. The patient's medical history, including any recent infections or surgeries, should also be reviewed.
2. Rapid laboratory testing: Blood cultures and other laboratory tests, such as complete blood counts and serum lactate levels, should be performed promptly to confirm the presence of an infection and assess its severity.
3. Administration of antibiotics: Broad-spectrum antibiotics should be administered as soon as possible, ideally within the first hour of recognition of sepsis. The choice of antibiotics should be guided by the suspected source of the infection and the patient's allergies an

The large language model (LLM) generated an answer to the posed question based on the data utilized during its training process. The answer outlines ten sequential steps for the management of sepsis. It concludes with a remark emphasizing the essential role of a critical care unit in the treatment of affected patients.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [23]:
response_llm_2 = response(query_2, max_tokens=MAX_TOKENS)
print(response_llm_2)

Llama.generate: prefix-match hit



Answer:

Appendicitis is a medical emergency that requires prompt treatment. The common symptoms of appendicitis include:

1. Severe pain in the abdomen, usually starting near the belly button and then moving to the lower right side of the abdomen.
2. Nausea and vomiting.
3. Loss of appetite.
4. Fever.
5. Abdominal tenderness and guarding (muscle tension).
6. Abdominal swelling.
7. Diarrhea or constipation.

If you suspect that you or someone else has appendicitis, it is essential to seek medical attention immediately. Appendicitis cannot be cured via medicine; instead, surgery is required to remove the inflamed appendix. The surgical procedure most commonly used to treat appendicitis is an appendectomy, which involves removing the inflamed appendix through a small incision in the abdomen.

There are two types of appendectomies: open and laparoscopic. Open appendectomy involves a larger incision in the abdomen, while laparoscopic appendectomy involves several small incisions and the u

The large language model (LLM) structured its response into two distinct sections. The first section enumerates seven symptoms associated with appendicitis. The second section states that no pharmacological treatment exists for the condition, indicating that surgical intervention is the sole therapeutic option, and offers further details regarding the surgical procedure. The LLM addressed both questions drawing exclusively on the data acquired during its training.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [24]:
response_llm_3 = response(query_3, max_tokens=MAX_TOKENS)
print(response_llm_3)

Llama.generate: prefix-match hit



Sudden patchy hair loss, also known as alopecia areata, is a common condition that can cause localized bald spots on the scalp. While there is no cure for this condition, there are several effective treatments and solutions that can help promote hair growth and reduce the size of the bald spots. Here are some possible causes and treatment options:

Causes:

1. Autoimmune disorder: Alopecia areata is believed to be caused by an autoimmune response, where the body's immune system mistakenly attacks healthy hair follicles, leading to hair loss.
2. Hormonal imbalance: Hormonal changes, such as those that occur during pregnancy or menopause, can trigger alopecia areata.
3. Stress: Physical or emotional stress can contribute to the development of alopecia areata.
4. Genetics: Alopecia areata can run in families, suggesting a genetic component.

Treatment options:

1. Corticosteroid injections: Injecting corticosteroids into the affected area can help reduce inflammation and promote hair gro

The large language model (LLM) successfully addressed both questions, organizing its response into two distinct sections. The first section outlines the causes of alopecia, while the second presents ten potential treatment options.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [25]:
response_llm_4 = response(query_4, max_tokens=MAX_TOKENS)
print(response_llm_4)

Llama.generate: prefix-match hit



The treatment plan for a person with a physical injury to brain tissue will depend on the severity and location of the injury, as well as the individual's overall health and medical history. Here are some common treatments that may be recommended:

1. Medications: To manage symptoms such as pain, inflammation, and anxiety.
2. Rehabilitation therapy: To help regain lost function and improve cognitive and physical abilities. This may include physical therapy, occupational therapy, speech therapy, and cognitive therapy.
3. Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or repair damaged tissue.
4. Lifestyle changes: To help manage symptoms and improve overall health, such as getting regular exercise, eating a healthy diet, and getting enough sleep.
5. Cognitive rehabilitation: To help improve cognitive function and memory.
6. Neuropsychological testing: To assess the extent of brain damage and develop a treatment plan tailored to the individual's needs.

The large language model (LLM) generated a response outlining the management of a patient with a physical brain injury. The output is clearly structured, featuring an introductory statement followed by an organized enumeration of treatment options.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [26]:
response_llm_5 = response(query_5, max_tokens=MAX_TOKENS)
print(response_llm_5)

Llama.generate: prefix-match hit



A person who has fractured their leg during a hiking trip requires immediate medical attention to prevent further complications and ensure proper healing. Here are some necessary precautions and treatment steps:

1. Stop all activities: The first step is to stop all activities, especially weight-bearing exercises, to avoid exacerbating the injury. This includes hiking, walking, or any other physical activity that may put pressure on the affected leg.
2. Seek medical attention: It is essential to seek medical attention as soon as possible, preferably within 24-48 hours of the injury. A healthcare professional will assess the severity of the fracture and provide appropriate treatment.
3. Immobilize the leg: To prevent further damage and promote healing, the affected leg should be immobilized using a splint or cast. This will help keep the bones in place and reduce pain.
4. Manage pain: Pain management is crucial to ensure the person can rest comfortably and undergo treatment without dis

The large language model (LLM) generated a response addressing the management of a fractured leg sustained during a hiking trip. The output includes both introductory and concluding statements and presents nine sequential actions encompassing treatment, care, and recovery.

## 7 - Question Answering using LLM with Prompt Engineering

In this section, the same set of questions will be posed to the LLM, with each question employing a different prompt engineering approach. For this purpose, a prompt builder function—defined in the utility functions section of this notebook—has been implemented. This function incorporates both a system message and a user message to construct the prompt, which is then passed to the response function responsible for generating the LLM’s output.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [27]:
# instruction prompt
query_eng_1 = """
Your task is to provide a comprehensive, evidence-based protocol for managing sepsis in a critical care unit.
Your response must be less than 512 words
Begin your response now.
"""

In [28]:
user_input = llama_prompt_builder(system_prompt,query_eng_1)
response_eng_1 = response(user_input,max_tokens=512, temperature=2, top_k=80, top_p=1)
print(response_eng_1)

Llama.generate: prefix-match hit


 Protocol for Managing Sepsis in the Critical Care Unit A standardized, evidence-based sepsis protocol is paramount... Sepcertain situations as gupta Proceedure and/Brown University Cente for biovettual Inameduares). C. Empiric Antimucobacterial Medication Driubbed throughout these procedures arto bemonitorvital sifns of severeklife-threeting 15(-points;6r hemodiynamicsand respidony ptotected sepsi;2d/1  reapiratorsupport if ptimateindicat ed. IV Fluid Stratum Bund Blood;NS), along  s fluid s overevance or dobutexorshop erive shock refrotocol should aim to ideal Patient Comfurter(if necessary), mane possible airwa Ed; an eshetic ed with siderable  clinlicallcnicsoope of stus, pffering mocules ptoclract a clinigc dclepmension: D. Pain 
Control, Restraint s sedat ion. Protocol form aninguistic method so fllor stalled cecuts unsing these principals; anydevuise clcarificadods ar tthe lg-term benenitssaveing inut the near-te rvmed vantages) of icu pr im , s or ptims e reassure our sepsis pa

For the first query, an instruction-based prompt was employed. The prompt clearly defines the task, incorporates a constraint of fewer than 512 words, and directs the LLM to respond immediately.

The parameters for the model are set to generate more creativity and variability in the response. The temperature is set to 2, top_k to 80 and top_p to 1.

The model failed to produce a coherent sepsis protocol. The output contained nonsensical content, invented terms, misspellings, and errors, suggesting excessive randomness.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [29]:
# Chain of thought prompt
query_eng_2= """
Your task is to answer the following question using a step-by-step reasoning approach, ensuring clarity, accuracy, and logical progression:
**Question:** What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

Begin by identifying the typical symptoms of appendicitis. Consider both early and advanced signs, and explain how they help differentiate appendicitis from other abdominal conditions.

Next, evaluate whether appendicitis can be treated with medication alone. Discuss the role of antibiotics in uncomplicated cases. Include the risks of recurrence and criteria for selecting patients for non-surgical treatment.

Then, if surgery is required, describe the standard surgical procedures used to treat appendicitis. Compare **open appendectomy** and **laparoscopic appendectomy**, highlighting their indications, benefits, and recovery profiles.

Begin your chain of thought now.
"""

In [30]:
user_input = llama_prompt_builder(system_prompt,query_eng_2)
response_eng_2 = response(user_input,max_tokens=MAX_TOKENS, top_k=30, top_p=0.3)
print(response_eng_2)

Llama.generate: prefix-match hit



As a professional medical assistant, I'll be happy to help answer your question about appendicitis symptoms and treatment options. Here's the step-by-step reasoning approach:

1. Typical Symptoms of Appendicitis:

Early signs and symptoms of appendicitis may include:

* Sudden, severe pain in the lower right abdomen that begins around the navel and then moves to the lower right abdomen
* Nausea and vomiting
* Loss of appetite
* Fever (usually less than 101°F)
* Abdominal tenderness and guarding (muscle tension)
* Abdominal swelling

Advanced signs and symptoms may include:

* Severe pain in the lower right abdomen that worsens with movement or coughing
* Abdominal rigidity (stiffness)
* Inability to pass gas
* Constipation or diarrhea
* Palpable (able to be felt) tenderness in the abdomen

These symptoms can help differentiate appendicitis from other abdominal conditions, such as a ruptured ovarian cyst or a blockage in the intestines. However, it's essential to seek medical attention

In this query, a chain-of-thought prompt was employed to guide the model's response. The prompt explicitly instructs the model to adopt a step-by-step reasoning approach, with the aim of fostering logical coherence and clarity. This structure is designed to prevent premature conclusions and superficial analysis. The model organizes its response into three distinct sections. First, it identifies the early symptoms of appendicitis, followed by a description of more advanced clinical manifestations. In the second section, the model evaluates the potential for medical management, noting that antibiotic therapy may be effective in uncomplicated cases, whereas surgical intervention is typically required for more severe presentations. The final section outlines two standard surgical procedures for treating appendicitis—open appendectomy and laparoscopic appendectomy—highlighting their respective characteristics and clinical considerations.

The parameters for the LLM response, temperature set to 0, top_p set to 0.3 and top_k set to 30 will guide the model to produce a more deterministic, focused and coherent output.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [31]:
# Structured + Template based prompt
query_eng_3 = """
Your task is to answer the following question in a structured and clinically accurate manner.

<<QUESTION>>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
<</QUESTION>>

<<RESPONSE FORMAT>>
Please respond using the following structure:
1. **Definition and Description**
   - Briefly define sudden patchy hair loss and describe its typical presentation.
2. **Possible Causes**
   - List and explain the most common causes, including:
   - Alopecia areata
   - Tinea capitis
   - Trichotillomania
   - Autoimmune or hormonal conditions
3. **Effective Treatments**
   - Provide evidence-based treatments, including:
   - Topical and oral medications (e.g., corticosteroids, minoxidil)
   - Immunotherapy or biologics (if applicable)
   - Lifestyle and nutritional interventions
   - Psychological support (if stress-related)
4. **Prognosis and Follow-Up**
   - Discuss expected outcomes, recurrence risk, and when to seek specialist care.
<</RESPONSE FORMAT>>

<<STYLE GUIDELINES>>
- Use clear, professional language suitable for both clinicians and informed patients.
- Avoid speculation or unsupported claims.
- Keep the tone empathetic and informative.
<</STYLE GUIDELINES>>

Begin your response now.

"""

In [32]:
user_input = llama_prompt_builder(system_prompt,query_eng_3)
response_eng_3 = response(user_input,max_tokens=MAX_TOKENS, temperature=0.7, top_p=0.9, top_k=50)
print(response_eng_3)

Llama.generate: prefix-match hit



As a professional medical assistant, I will provide a structured and clinically accurate response to address sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and explore the possible causes and effective treatments.

1. Definition and Description: 
Sudden patchy hair loss, also known as alopecia areata, is characterized by well-defined, round or oval-shaped bald patches on the scalp. The affected areas may be smooth and hairless, with no inflammation or scaling. This condition can occur at any age but is more common in younger adults and children.

2. Possible Causes: 
The exact cause of sudden patchy hair loss remains unclear, but several factors are associated with its development. These include:
   - Alopecia areata: An autoimmune condition that causes the immune system to attack healthy hair follicles, leading to hair loss.
   - Tinea capitis (ringworm): A fungal infection of the scalp that can cause patchy hair loss and inflammation.
   - Trichotilloma

In this instance, a structured, template-based prompt was employed to guide the model’s output. The prompt delineated a multipart response framework, explicitly instructing the model to organize its answer into four distinct sections:
- Definition and Description
- Possible Causes
- Effective Treatments
- Prognosis and Follow-Up

The prompt incorporated clearly labeled components—namely, QUESTION, RESPONSE FORMAT, and STYLE GUIDELINES—which served as a scaffold for the model’s generation process. The model demonstrated full compliance with the prescribed four-part structure and consistently adhered to the stylistic conventions outlined in the prompt.

The parameters passed to the model generated a balanced response. With a temperature of 0.7, top_p of 0.9 and top_k of 50.  

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [33]:
# Few-shot Prompt
query_eng_4 = """
Your task is to answer questions about brain injuries using structured, evidence-based reasoning. Follow the format shown in the examples below.

---

**Example 1**
**Question:** What are the symptoms of a mild concussion?
**Answer:**
1. Headache and dizziness
2. Temporary confusion or memory loss
3. Nausea or vomiting
4. Sensitivity to light or noise
5. Fatigue and difficulty concentrating
These symptoms typically resolve within days to weeks. Medical evaluation is recommended to rule out more serious injury.

---

**Example 2**
**Question:** What are the common causes of traumatic brain injury?
**Answer:**
1. Falls (especially in older adults and children)
2. Motor vehicle accidents
3. Sports injuries (e.g., football, boxing)
4. Assaults or blunt trauma
5. Blast injuries (common in military settings)
These injuries can be classified as penetrating or non-penetrating, and severity ranges from mild to severe.

---

Now answer the following question using the same format:
**Question:** What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
"""

In [34]:
user_input = llama_prompt_builder(system_prompt,query_eng_4)
response_eng_4 = response(user_input,max_tokens=MAX_TOKENS, top_k=100, top_p=1)
print(response_eng_4)

Llama.generate: prefix-match hit



**Answer:**

Treatments for physical injuries to brain tissue depend on the severity and location of the injury. Here are some common recommendations:

1. Medications: Pain relievers, anti-seizure drugs, and antidepressants may be prescribed to manage symptoms and prevent complications.
2. Rehabilitation therapy: Physical, occupational, and speech therapy can help restore lost functions and improve cognitive abilities.
3. Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or repair damaged tissue.
4. Lifestyle modifications: Patients may need to make adjustments to their daily routine, such as avoiding stressful activities, taking regular breaks, and prioritizing self-care.
5. Cognitive training: Techniques like cognitive-behavioral therapy (CBT) can help individuals with brain injuries manage memory loss, attention issues, and other cognitive challenges.
6. Assistive technology: Devices such as wheelchairs, walkers, and communication aids may be necessa

This query was constructed using a few-shot prompting strategy, wherein an explicit instruction was paired with multiple examples to guide the language model's response. The prompt contained two illustrative cases that exemplified the desired format, tone, and depth of content. These exemplars functioned as implicit procedural cues, enabling the model to abstract the underlying structure and apply it effectively to the target question. The model successfully adhered to the inferred schema, producing a coherent answer that enumerated various treatment modalities and elaborated on each within the expected informational framework.

The parameters will try to combine a temeprature of 0 to hace a deterministic response, but allow some freedom and creativity with a high top_k of 100 and top_p of 1 looking to have some balance. The model was able to generate a response that makes sense and is readable, but has some vague sentences, and doesn't expand explaining medical concepts.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [35]:
#Stepwise prompt
query_eng_5 = """
Your task is to answer questions about injuries sustained during outdoor activities using a step-by-step format. Follow the structure below to ensure clarity, safety, and completeness.

---

**Question:** What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

**Answer:**

**Step 1: Immediate Assessment and Safety**
- Indicate how to check and assess a fracture

**Step 2: Emergency Stabilization**
- Indicate how to stabilize the leg if possible

**Step 3: Pain and Shock Management**
- Indicate how to treat pain produced by a fracture

**Step 4: Evacuation Planning**
- Indicate how to move the person

**Step 5: Medical Treatment**
- indicate how to transport to a hospital

**Step 6: Recovery and Rehabilitation**
- Indicate how to recover from the injury

**Step 7: Return to Activity**
- Indicate how to return to activity

---

Now answer the question using this stepwise format.

"""

In [36]:
user_input = llama_prompt_builder(system_prompt,query_eng_5)
response_eng_5 = response(user_input,max_tokens=MAX_TOKENS, temperature=2, top_k=30, top_p=0.2)
print(response_eng_5)

Llama.generate: prefix-match hit



**Answer:**

**Step 1: Immediate Assessment and Safety**
If a person has fractured their leg during a hiking trip, it is essential to assess the injury immediately and prioritize safety. Here are some steps to follow:

* Check for signs of shock or hypovolemia (low blood volume) such as pale skin, cool to the touch, fast heart rate, shallow breathing, or decreased level of consciousness. If present, call 911 or your local emergency number immediately.
* Check for other injuries, such as head trauma, spinal injury, or soft tissue damage.
* Ensure the person is in a safe location, away from any hazards or further harm.
* Use a first aid kit to treat any minor injuries, such as cleaning and dressing wounds.

**Step 2: Emergency Stabilization**
If possible, stabilize the leg using the following steps:

* Immobilize the affected limb using a splint or other available materials to prevent further movement and reduce pain.
* Use a cold compress or ice pack wrapped in a towel to reduce swelli

For this query, I used a stepwise prompt aimed at helping the model produce a complete and organized response. The prompt outlined seven key stages—starting with assessment and stabilization, moving through pain management, evacuation, and medical care, and ending with recovery and return to physical activity. Each step was presented in a logical order to guide the model’s output. The model followed the structure well, describing relevant actions within each phase and offering clinically informed details that matched the intent of each label.

The parameters pass to the model are set to allow for more creative response with a temperature of 2, but regulate with top_p set to 0.2 and top_k set to 30.

Comapring against the response of the first query that also used a temperature of 2, the response for this last question is readable, and makes sense. So regulating with top_p and top_k help the model to generate a creative but readable response.

**Prompt Engineering and Tuning**

The model successfully adhered to various prompt engineering techniques, demonstrating its ability to respond effectively to structured examples and templates. This approach enhances output quality and facilitates more nuanced responses.

Striking an optimal balance in model parameters is crucial. Excessively high values for temperature, top-p, and top-k can introduce unwarranted randomness, as evidenced in the first query. Conversely, overly restrictive settings may yield deterministic, repetitive, and constrained outputs.

## 8 - Data Preparation for RAG

### Loading the Data

Within the Library Importing section, Google Drive has already been mounted and the variable FILENAME has been initialized with the file path referencing the Merck Manual. The subsequent code block is responsible for loading the corresponding PDF document into the environment.

In [37]:
# Load the PDF document using PyMuPDFLoader
pdf_loader = PyMuPDFLoader(FILENAME)
pages = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [38]:
# Print the first 5 pages of the PDF document
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(pages[i].page_content,end="\n")
    print("---" * 30, end="\n")

Page Number : 1
fernando88@gmail.com
EPFAT4LO8B
eant for personal use by fernando88@gm
shing the contents in part or full is liable 

------------------------------------------------------------------------------------------
Page Number : 2
fernando88@gmail.com
EPFAT4LO8B
This file is meant for personal use by fernando88@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

------------------------------------------------------------------------------------------
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......

The initial five pages of the document contribute minimal substantive content, as they primarily consist of the manual’s cover and index. Additionally, each page contains a recurring footer. To enhance data quality and reduce extraneous elements, it would be advantageous to remove both the preliminary non-informative pages and the persistent footer across the document.

#### Checking the number of pages

In [39]:
# document length
len(pages)

4114

The manual has 4,114 pages

### Data Cleaning

The initial ten pages of the manual consist solely of the cover and index, and therefore do not contribute meaningful content for the RAG model. To ensure that only relevant information is incorporated into the vector database, these pages will be excluded from the preprocessing pipeline.

In [40]:
# delete the first 10 pages of the manual
del pages[:11]

In [41]:
# Verify that the first page is the one after the index.
pages[0].page_content

'THE MERCK MANUAL OF DIAGNOSIS AND THERAPY - 19th Ed. (2011)\nFront Matter\nTitle Page\nThe Merck Manual Of Diagnosis and Therapy, Nineteenth Edition\nRobert S. Porter MD, Editor-in-Chief\nJustin L. Kaplan MD, Senior Assistant Editor\nEditorial Board\nRichard K. Albert MD\nMarjorie A. Bowman MD, MPA\nGlenn D. Braunstein MD\nSidney Cohen MD\nLinda Emanuel PhD\nJan Fawcett MD\nEugene P. Frenkel MD\nSusan L. Hendrix DO\nMichael Jacewicz MD\nMatthew E. Levison MD\nJames Jeffrey Malatack MD\nBrian F. Mandell MD, PhD\nGerald L. Mandell MD\nJudith S. Palfrey MD\nAlbert A. Rundio Jr., PhD\nDavid A. Spain MD\nPaul H. Tanser MD\nMichael R. Wasserman MD\nMerck Sharp & Dohme Corp., A Subsidiary of Merck & Co., Inc.\nWhitehouse Station, NJ\n2011\nCopyright Page\nEditorial and Production Staff\nExecutive Editor: Keryn A.G. Lane\nSenior Staff Writers : Susan T. Schindler\n                                  Susan C. Short\nStaff Editor : Michelle A. Steigerwald\nSenior Operations Manager : Diane C. Zen

Each page of the manual includes a recurring footer that lacks semantic relevance for the model. To prevent this non-informative content from influencing the quality of the embeddings, it is better to remove the footer prior to storing the text chunks in the vector database.

In [42]:
# Remove footer from every page.
pages = remove_footer(pages, footer)

In [43]:
# Observe that the footer is no longer in a page.
pages[20].page_content

"Neonatal-Perinatal Medicine, University of\nMichigan, C.S. Mott Children's Hospital\nApproach to the Care of Normal Infants and\nChildren; Perinatal Physiology; Caring for\nSick Children and Their Families\nWILLIAM J. COCHRAN, MD\nVice Chairman, Department of Pediatrics,\nGeisinger Clinic, Danville, PA\nGastrointestinal Disorders in Neonates and\nInfants; Congenital Gastrointestinal Anomalies\nALAN S. COHEN, MD\nDistinguished Professor of Medicine (Emeritus),\nConrad Wessolhoeft Professor of Medicine,\nBoston University School of Medicine; Editor-\nin-Chief, Amyloid: The Journal of Protein\nFolding Disorders\nAmyloidosis\nROBERT B. COHEN, DMD\nClinical Associate Professor of Dentistry and\nPractice Coordinator, Tufts University School\nof Dental Medicine\nApproach to Dental and Oral Symptoms\nSIDNEY COHEN, MD\nProfessor of Medicine and Director, Research\nPrograms, Thomas Jefferson University\nSchool of Medicine\nGastritis and Peptic Ulcer Disease; Bezoars\nand Foreign Bodies; Hepatit

In [44]:
# Document length after removing the footer and  removing non-informative pages.
len(pages)

4103

The document currently comprises 4,104 pages, and upon inspecting a representative sample—such as page 20—it is evident that the recurring footer previously present across pages has been successfully removed.

### Data Chunking

Following the loading and cleaning of the document, the initial operation involves segmenting the text into discrete chunks. The cl100k_base encoding was selected due to its compatibility with widely-used large language models, such as OpenAI's GPT-4. A chunk size of 512 tokens was configured to strike a balance between semantic cohesion and model constraints, as larger chunks risk exceeding the context window and triggering inference errors. To mitigate context discontinuity between segments, a token overlap of 25 has been implemented—this promotes smoother transitions while minimizing redundant content across chunk boundaries.

In [45]:
# Initialize the text splitter to chunk the document into smaller pieces.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=25
)

In [46]:
# Split the document into smaller chunks for processing.
chunks = text_splitter.split_documents(pages)

In [47]:
# Print first 5 chunks of the document.
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(chunks[i].page_content,end="\n")
    print("---" * 30, end="\n")

Page Number : 1
THE MERCK MANUAL OF DIAGNOSIS AND THERAPY - 19th Ed. (2011)
Front Matter
Title Page
The Merck Manual Of Diagnosis and Therapy, Nineteenth Edition
Robert S. Porter MD, Editor-in-Chief
Justin L. Kaplan MD, Senior Assistant Editor
Editorial Board
Richard K. Albert MD
Marjorie A. Bowman MD, MPA
Glenn D. Braunstein MD
Sidney Cohen MD
Linda Emanuel PhD
Jan Fawcett MD
Eugene P. Frenkel MD
Susan L. Hendrix DO
Michael Jacewicz MD
Matthew E. Levison MD
James Jeffrey Malatack MD
Brian F. Mandell MD, PhD
Gerald L. Mandell MD
Judith S. Palfrey MD
Albert A. Rundio Jr., PhD
David A. Spain MD
Paul H. Tanser MD
Michael R. Wasserman MD
Merck Sharp & Dohme Corp., A Subsidiary of Merck & Co., Inc.
Whitehouse Station, NJ
2011
Copyright Page
Editorial and Production Staff
Executive Editor: Keryn A.G. Lane
Senior Staff Writers : Susan T. Schindler
                                  Susan C. Short
Staff Editor : Michelle A. Steigerwald
Senior Operations Manager : Diane C. Zenker
Senior Project 

In [48]:
len(chunks)

8179

The number of chunks is almost the doble in size to the original 4104 pages of the document.

### Embedding

In [49]:
# Initialize the embedding model for vectorization.
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [50]:
embedding_1 = embedding_model.embed_query(chunks[0].page_content)
embedding_2 = embedding_model.embed_query(chunks[1].page_content)

In [51]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

The thenlper/gte-large model was selected as it is a powerful transformer-based tool for generating high-quality text embeddings. It’s designed to understand and capture semantic meaning really well, which makes it a solid choice for tasks like document search, finding similar pieces of text, or grouping related content together in NLP workflows.

Each embedding generated by the model is a 1,024 dimensional vector.

### Vector Database

In [52]:
# Create a Chroma vector store to store the document chunks and their embeddings.
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [53]:
# Create a Chroma vector store from the document chunks and their embeddings.
vectostore = Chroma.from_documents(
    chunks,
    embedding_model,
    persist_directory=out_dir
)


In [54]:
# Load the vector store from the persisted directory.
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [55]:
# Details of the vector store embeddings
vectostore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False)

In [56]:
# Perform a similarity search on the vector store using a keywords.
vectorstore.similarity_search("Hypertension blood preasure artery",k=3)

[Document(page_content='The Merck Manual of Diagnosis & Therapy, 19th Edition\nChapter 208. Arterial Hypertension\n2228', metadata={'author': '', 'creationDate': 'D:20120615054440Z', 'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/drive/MyDrive/University of Texas/Medical Assistant Project/data/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': 'D:20250725213725Z', 'page': 2237, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/drive/MyDrive/University of Texas/Medical Assistant Project/data/medical_diagnosis_manual.pdf', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114, 'trapped': ''}),
 Document(page_content='Chapter 208. Arterial Hypertension\nIntroduction\nHypertension is sustained elevation of resting systolic BP (≥ 140 mm Hg), diastolic BP (≥ 90 mm\nHg), or both. Hypertension with no known cause (primary; formerly, essential hypertension) is\nmost common. H

The  document chunks has been successfully indexed within the vector store, enabling semantic retrieval through similarity-based querying. Each chunk is represented by a 1,024-dimensional embedding vector produced by the selected model, which effectively encodes the underlying meaning of the text for accurate and context-aware matching.

The similarity search contained the keywords "Hypertension", "blood preasure" and "artery". it return the top 3 pages with this keywords.

### Retriever

To facilitate semantic retrieval based on vector similarity, a retriever object is required. Configured with k = 3, the retriever selects the top three most relevant results in response to a given query. This constraint not only enhances the precision of model responses but also helps prevent the RAG prompt from exceeding the model’s context window.

In [57]:
# Create a retriever from the vector store for similarity search.
retriever = vectostore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [58]:
# Example user input to retrieve relevant documents.
user_input = "What treatment options are available for managing hypertension?"
rel_docs = retriever.get_relevant_documents(user_input)

In [59]:
# print the retrieved documents
rel_docs

[Document(page_content="diagnosis.\nPatients with labile, significantly elevated BP and symptoms such as headache, palpitations, tachycardia,\nexcessive perspiration, tremor, and pallor are screened for pheochromocytoma (eg, by measuring plasma\nfree metanephrines—see p. 802).\nPatients with symptoms suggesting Cushing's syndrome, a connective tissue disorder, eclampsia, acute\nporphyria, hyperthyroidism, myxedema, acromegaly, or CNS disorders are evaluated (see elsewhere in\nTHE MANUAL).\nPrognosis\nThe higher the BP and the more severe the retinal changes and other evidence of target-organ\ninvolvement, the worse is the prognosis. Systolic BP predicts fatal and nonfatal cardiovascular events\nbetter than diastolic BP. Without treatment, 1-yr survival is < 10% in patients with retinal sclerosis, cotton-\nwool exudates, arteriolar narrowing, and hemorrhage (grade 3 retinopathy), and < 5% in patients with the\nsame changes plus papilledema (grade 4 retinopathy). CAD is the most common c

In [60]:
# number of retrieved documents
len(rel_docs)

3

In [61]:
# Length of the first retrieved document.
len(rel_docs[0].page_content)

2143

The retriver was able to look for the pages that mention how to treat hypertension. It returned only the top 3 documents.

## 8 - Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [62]:
response_rag_1 = generate_rag_response(query_1, max_tokens=MAX_TOKENS)
print(response_rag_1)

Llama.generate: prefix-match hit


Based on the information provided in the context, the protocol for managing sepsis in a critical care unit includes:

1. Early recognition of sepsis and prompt administration of antibiotics, with a regimen that includes gentamicin or tobramycin and a third-generation cephalosporin.
2. Monitoring of vital signs, including temperature, blood pressure, pulse rate, and respiration rate, as well as quantification of all fluid intake and output.
3. Measurement of electrolytes and CBC daily, and measurement of Mg, phosphate, and Ca levels in patients with arrhythmias.
4. Use of point-of-care testing for blood tests, including electrolytes and CBC, at the patient's bedside or unit.
5. Draining of abscesses and surgical excision of necrotic tissues.
6. Maintenance of normal blood glucose levels through continuous IV insulin infusion, titrated to maintain glucose between 80 to 110 mg/dL.
7. Fluid resuscitation with 0.9% saline until CVP reaches 8 mm Hg or PAOP reaches 12 to 15 mm Hg, and monitor

The RAG response demonstrates a higher degree of clinical specificity, incorporating specific medication such as gentamicin and dopamine, as well as clearly defined  metrics like glucose regulation thresholds. Compared to responses generated solely from the model’s pre-trained data, this output reflects a more contextually grounded and protocol-aligned representation of sepsis management practices.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [63]:
response_rag_2 = generate_rag_response(query_2, max_tokens=MAX_TOKENS)
print(response_rag_2)

Llama.generate: prefix-match hit


Based on the provided context, here are the answers to your questions:

###Symptoms of Appendicitis:

The common symptoms of appendicitis include:

1. Abdominal pain (often severe and located near the navel)
2. Nausea and vomiting
3. Loss of appetite
4. Fever
5. Abdominal tenderness and guarding (muscle tension)
6. Abdominal swelling
7. Diarrhea or constipation

It is important to note that the symptoms of appendicitis can vary, and not everyone will experience all of them. Additionally, the symptoms can be similar to other conditions, such as a ruptured ovarian cyst or a twisted bowel, so it is essential to seek medical attention if you suspect appendicitis.

###Can Appendicitis be Cured via Medicine?

No, appendicitis cannot be cured via medicine. The only effective treatment for appendicitis is surgical removal of the inflamed appendix. Delaying surgery can lead to complications such as abscesses, peritonitis, and potentially life-threatening infections.

###Surgical Procedure for T

The response generated via retrieval-augmented generation (RAG) demonstrates a clear improvement in accuracy and specificity. It clearly states that appendicitis can’t be cured with medicine—which corrects earlier answers that got that part wrong. Plus, it goes a step further by explaining how the surgery works, mentioning anesthesia, how long the procedure usually takes, and even outlining what happens afterward. Overall, it feels grounded in real medical protocol and hits the mark with domain-specific details.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [64]:
response_rag_3 = generate_rag_response(query_3, max_tokens=MAX_TOKENS)
print(response_rag_3)

Llama.generate: prefix-match hit


Based on the information provided in the context, the following are the effective treatments or solutions for addressing sudden patchy hair loss:

1. Minoxidil: Topical minoxidil (2% for women and 2% or 5% for men) can prolong the anagen growth phase and gradually enlarge miniaturized follicles into mature terminal hairs.
2. Finasteride: Finasteride inhibits the 5α-reductase enzyme, blocking conversion of testosterone to dihydrotestosterone, and is useful for male-pattern hair loss.
3. Corticosteroids: Oral or topical corticosteroids may be effective in treating sudden patchy hair loss due to alopecia areata, lichen planopilaris, or chronic cutaneous lupus lesions.
4. Immunomodulators: Topical immunotherapy with diphencyprone or squaric acid dibutylester may be useful in treating alopecia areata.
5. Surgical options: Follicle transplant, scalp flaps, and alopecia reduction may be considered for patients who desire a more permanent solution.

Possible causes of sudden patchy hair loss i

The RAG-generated response demonstrates improved detail and completeness compared to earlier outputs. It provides specific dosage information for topical Minoxidil—differentiating between male and female use—and expands the treatment options by including Finasteride and various surgical interventions. Additionally, the list of possible causes is more comprehensive, incorporating conditions such as lichen planopilaris, traction alopecia, and chronic cutaneous lupus lesions. Overall, the response exhibits greater clinical depth and broader diagnostic coverage, enhancing its relevance and specificity.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [65]:
response_rag_4 = generate_rag_response(query_4, max_tokens=MAX_TOKENS)
print(response_rag_4)

Llama.generate: prefix-match hit


Based on the information provided in the context, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, are:

1. Supportive care to prevent systemic complications due to immobilization, such as pneumonia, UTI, thromboembolic disease, and to provide good nutrition.
2. Prevention of pressure ulcers.
3. Rehabilitation therapy to maximize functional recovery, including cognitive therapy for patients with severe cognitive dysfunction.
4. Surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
5. Maintenance of adequate brain perfusion and oxygenation in the first few days after the injury.

It is important to note that the specific treatment will depend on the level and extent of the injury, as well as the presence of any complications. Early intervention by rehabilitat

The RAG response demonstrates superior clinical specificity by emphasizing acute interventions, such as intracranial monitoring, decompressive procedures, and hematoma evacuation. In contrast, the prompt-engineered output adopts a broader, multi-dimensional framework that includes psychological support, lifestyle adjustments, assistive technologies, and community-based resources. While the surgical guidance in the RAG output is more technically detailed, the prompt-engineered version presents surgery in general terms. Overall, the RAG approach is tailored toward immediate clinical stabilization, whereas the prompt-engineered response centers on longitudinal rehabilitation and functional reintegration.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [66]:
response_rag_5 = generate_rag_response(query_5, max_tokens=MAX_TOKENS)
print(response_rag_5)

Llama.generate: prefix-match hit


Based on the provided context and question, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. Rewarming: The affected area should be rewarmed rapidly by totally immersing the leg in water that is tolerably warm to the touch (40-42°C, ideally about 40.5°C). Avoid using dry heat sources like fire or heating pads as they may cause burns.
2. Pain management: Patients are given analgesics, including opioids, to manage pain during rewarming.
3. Elevation: The injured limb should be elevated above the heart for the first two days in a position that allows gravity to help drain edema fluid and minimize swelling.
4. Immobilization: Immobilization is helpful in decreasing pain and facilitating healing by preventing further injury. Joints proximal and distal to the injury should be immobilized. A cast or splint can be used for fractures or other injuries that require weeks of immobilization.
5. Hygiene: Good hygiene is import

The RAG response exhibits greater medical specificity by concentrating on post-injury clinical measures such as rewarming techniques, analgesic management, limb elevation, cast hygiene, and patient monitoring. In contrast, the Prompt-Engineer response is oriented toward field triage priorities—emphasizing immediate stabilization, shock assessment, and evacuation logistics. Notably, the RAG output delves deeper into complication prevention, highlighting conditions such as compartment syndrome, infection risk, and muscle atrophy, whereas the Prompt-Engineer approach offers only limited coverage of acute systemic risks like hypovolemia. Overall, the RAG response demonstrates superior alignment with hospital-based care protocols and incorporates more granular guidance on cast care and thermal recovery.

## 9 - Fine-tuning

### Out of Context

In [67]:
query_6 = """
Explain what a large language model (LLM) is and how it generates responses.
"""
response_fine = generate_rag_response(query_6, max_tokens=MAX_TOKENS)
print(response_fine)

Llama.generate: prefix-match hit


I don't know. The context provided does not contain any information about large language models (LLMs) or how they generate responses. The context is focused on evaluating cognitive function in patients with suspected neurological disorders, specifically aphasia. Therefore, I cannot provide an explanation of LLMs or their response generation capabilities based on the given context.


The model responded correctly as the user query was about LLM and it has nothing to do with medical subject. As the context has no mention of LLM's it is correct that the model responded with "I don't know".

### Introduce Randomness

In [68]:
response_fine_1 = generate_rag_response(query_1, max_tokens=MAX_TOKENS, temperature=1.5)
print(response_fine_1)

Llama.generate: prefix-match hit


Based on the information provided in the context, the protocol for managing sepsis in a critical care unit includes:

1. Early recognition of sepsis with daily monitoring of vital signs, particularly temperature, blood pressure, pulse, and respiration rate.
2. Daily measurement of electrolytes and complete blood count (CBC) to help detect problems early.
3. Point-of-care testing using miniaturized, highly automated devices to do certain blood tests at the patient's bedside or unit.
4. Prompt empiric therapy with antibiotics based on the suspected source and clinical setting, and adjustment of antibiotic regimen based on culture and sensitivity results when available.
5. Maintenance of normalization of blood glucose levels through continuous IV insulin infusion titrated to maintain glucose between 80 to 110 mg/dL.
6. Frequent monitoring of systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, and electrolyte levels, as well as renal function and sublingual PCO2.
7.

In this query, the temperature setting was adjusted from 0 to 1.5 to introduce greater diversity in specificity and phrasing. The resulting output reflects enhanced elaboration and expanded clinical context, incorporating surgical, metabolic, and diagnostic dimensions. While this broader scope adds interpretive richness, the lower-temperature response remains superior for tightly structured ICU protocols, particularly in stabilizing septic patients and guiding antibiotic administration.

### Reduce Max Token

In [69]:
response_fine_2 = generate_rag_response(query_2, max_tokens=512, temperature=1.5)
print(response_fine_2)

Llama.generate: prefix-match hit


###Context

The context is acute appendicitis and its treatment.

###Question

What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

###Answer

Common symptoms of appendicitis include abdominal pain (often severe), anorexia, nausea, vomiting, fever, and abdominal tenderness. The pain typically begins near the navel and then moves to the lower right abdomen.

Appendicitis cannot be cured via medicine. Antibiotics may be used preoperatively to treat any concurrent infections, but they do not treat the underlying inflammation of the appendix. The surgical procedure for treating acute appendicitis is appendectomy, which involves removing the inflamed appendix.

Appendectomy can be performed either laparoscopically or open. Laparoscopic appendectomy is a minimally invasive technique that uses a camera and specialized instruments to remove the appendix through small incisions. Open appendectomy involv

In this configuration, the temperature was increased from 0 to 1.5 and the maximum token count reduced from 1024 to 512, resulting in a condensed output—306 words compared to the original 381. The shorter response is more concise due to the stricter token limit, while the elevated temperature promotes greater lexical variety and interpretive richness, notably through added references to monitoring parameters and procedural suitability. In contrast, the initial response (T=0, max_token=1024) maintains superior procedural clarity and adheres closely to standardized clinical documentation practices, making it preferable for protocol-driven retrieval and annotation workflows.

In [70]:
response_fine_3 = generate_rag_response(query_3, max_tokens=512, temperature=1.5)
print(response_fine_3)

Llama.generate: prefix-match hit


Based on the information provided in the context, the following are the effective treatments or solutions for addressing sudden patchy hair loss:

1. Minoxidil: Topical minoxidil (2% for women and 2% or 5% for men) can prolong the anagen growth phase and gradually enlarge miniaturized follicles into mature terminal hairs.
2. Finasteride: Finasteride inhibits the 5α-reductase enzyme, blocking conversion of testosterone to dihydrotestosterone, which is useful for male-pattern hair loss.
3. Corticosteroids: Oral or topical corticosteroids can be effective in treating sudden patchy hair loss due to alopecia areata.
4. Antifungals: Topical or oral antifungals may be useful in treating tinea capitis, a fungal infection that can cause hair loss.
5. Immunotherapy: Topical immunotherapy with diphencyprone or squaric acid dibutylester can be effective in treating alopecia areata.
6. Surgical options: Follicle transplant, scalp flaps, and alopecia reduction surgery may be considered for more seve

When comparing the two responses, there are key distinctions based on temperature and token constraints. The original response (Temperature = 0, Max Token = 1024) offers structured, detailed coverage with high procedural clarity. In contrast, the second response (Temperature = 1.5, Max Token = 512) is more concise, with 261 words compared to 285, and leverages the elevated temperature to introduce lexical diversity, broader diagnostic scope, and interpretive nuance. This shift in generation parameters results in an output better suited for educational contexts or patient-facing summaries, though with reduced depth in causative and therapeutic elaboration.

### Sampling - More Creative

In [71]:
response_fine_4 = generate_rag_response(query_4, max_tokens=512, temperature=1.5, top_p=1, top_k=100)
print(response_fine_4)


Llama.generate: prefix-match hit


Based on the information provided in the context, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function are:

1. Supportive care to prevent systemic complications due to immobilization such as pneumonia, UTI, thromboembolic disease, and to provide good nutrition.
2. Prevention of pressure ulcers.
3. Prevention of secondary disabilities such as joint contractures and pressure ulcers.
4. Family education.
5. Rehabilitation therapy including cognitive therapy, depending on the extent of the injury and development of complications.


The original response, using top-p 0.95 and top-k 50, sticks close to the source material. It's clean, efficient, and formal. The second response, dialed up with top-p 1 and top-k 100, takes more creative liberties. It’s a bit wordier, adds empathetic touches like “family education,” and explores long-term prognosis in a way that’s softer and more engaging. That’s because top-p 1 opens the door to more varied, lower-probability word choices, while the higher top-k offers a broader vocabulary pool, giving the model room to be more expressive and interpretive.

### Sampling - More Deterministic

In [72]:
response_fine_5 = generate_rag_response(query_5, max_tokens=512, temperature=0, top_p=0.5, top_k=40)
print(response_fine_5)

Llama.generate: prefix-match hit


Based on the provided context and question, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. Rewarming: The affected area should be rewarmed rapidly by totally immersing the leg in water that is tolerably warm to the touch (40-42°C, ideally about 40.5°C). Avoid using dry heat sources like fire or heating pads as they may cause burns.
2. Pain management: Patients are given analgesics, including opioids, to manage pain during rewarming.
3. Elevation: The injured limb should be elevated above the heart for the first two days in a position that allows gravity to help drain edema fluid and minimize swelling.
4. Immobilization: Immobilization is helpful in decreasing pain and facilitating healing by preventing further injury. Joints proximal and distal to the injury should be immobilized. A cast or splint can be used for fractures or other injuries that require weeks of immobilization.
5. Hygiene: Good hygiene is import

Although both responses were generated with deterministic temperature, changes to top-p and top-k had a noticeable impact. Response 1 (top-p 0.95, top-k 50) provided a complete, well-rounded treatment protocol with thoughtful clinical guidance and follow-up detail. Response 2 (top-p 0.5, top-k 40), constrained by tighter sampling and shorter max-token, delivered a structured but less expansive version, truncating before completing all considerations for recovery. The stricter top-p and lower top-k reduced linguistic variation and semantic breadth, which can be useful for applications needing controlled, consistent phrasing—but may limit expressive depth and completeness.

## 10 - Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Llama model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [73]:
# Generate groundedness and relevance ratings for the first query.
ground_1, relevance_1, answer_1 = generate_ground_relevance_response(query_1,max_tokens=MAX_TOKENS)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [74]:
# Print the groundedness rating
print(ground_1)

  Sure, I can help you with that! To evaluate the answer, we need to follow these steps:

Step 1: Check if the answer is derived only from the information presented in the context.

The answer provides a detailed protocol for managing sepsis in a critical care unit, which is based on the information provided in the context. The answer mentions specific antibiotics, monitoring parameters, and fluid resuscitation strategies that are consistent with the information in the context. Therefore, we can conclude that the answer is derived only from the information presented in the context.

Step 2: Evaluate the extent to which the metric is followed.

The answer provides a comprehensive protocol for managing sepsis in a critical care unit, which covers all the key aspects of sepsis management mentioned in the context. The answer includes specific antibiotics and monitoring parameters that are consistent with the information in the context. Therefore, we can conclude that the metric is followed

**Groundedness**

The groundedness evaluation indicates that the response fully meets the evaluation criteria by offering a complete and detailed protocol for sepsis management in a critical care setting. It aligns with all key aspects outlined in the context, resulting in a top score of 5.

In [75]:
# Print the relevance rating
print(relevance_1)

  Sure, I can help you with that! To evaluate the context as per the metric, we need to consider the following aspects:

1. Relevance: Does the answer address the main aspects of the question?
2. Completeness: Is the answer comprehensive and cover all the important aspects of the question?
3. Accuracy: Is the information in the answer accurate and up-to-date?
4. Clarity: Is the answer clear and easy to understand?
5. Relevance to the task: Does the answer help in completing the task or achieving the desired outcome?

Now, let's evaluate the context as per the metric:

1. Relevance: The answer addresses all the main aspects of the question, including the protocol for managing sepsis in a critical care unit, the importance of early recognition and prompt administration of antibiotics, and the need for close monitoring of vital signs and fluid resuscitation.
2. Completeness: The answer is comprehensive and covers all the important aspects of the question, including the use of vasopressors

**Relevance**

The evaluation marked the answer as relevant, comprehensive, and clinically accurate—drawing from current guidelines and research. The evaluation mentions that the answer covers critical components like early antibiotic administration, vital sign monitoring, vasopressor use, and organ function assessment, all while maintaining clarity and relevance to the task.
Overall, the evaluation gave a score of 4 to the answer.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [76]:
ground_2, relevance_2, answer_2 = generate_ground_relevance_response(query_2,max_tokens=MAX_TOKENS)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [77]:
print(ground_2)

  Sure, I'll evaluate the answer based on the metric provided.

Step 1: Evaluate the answer for relevance to the question.
The question asks about the common symptoms of appendicitis and the surgical procedure to treat it. The answer provides a comprehensive list of symptoms and describes the recommended surgical procedure, which is relevant to the question.

Step 2: Evaluate the answer for accuracy based on the context.
The answer accurately describes the symptoms of appendicitis and the surgical procedure to treat it, based on the information provided in the context. The answer also mentions the importance of prompt surgical intervention and the potential complications of appendicitis, which is accurate and relevant information.

Step 3: Evaluate the answer for completeness.
The answer provides a comprehensive list of symptoms and describes the recommended surgical procedure, which covers all aspects of the question. However, the answer does not provide any additional information or 

**Groundedness**

For the second query the Groundedness evaluation gives 3 scores based on the 3 steps mentioned in the prompt instructions. In the accuracy score it gives a score of 4 as there is no additional context. This lowers the overall score to 4.5

In [78]:
print(relevance_2)

  Sure, I'll be happy to help you evaluate the context and answer provided based on the given evaluation criteria.

Step 1: Evaluate the context for relevance

The context provides a detailed overview of appendicitis, including its symptoms, diagnosis, treatment options, and prognosis. The information is relevant to the question and covers all the important aspects of appendicitis. Therefore, the context adheres well to the metric.

Step 2: Evaluate the answer for relevance

The answer provides a concise summary of the common symptoms of appendicitis, the recommended surgical procedure, and the importance of prompt treatment. The information is relevant to the question and addresses all the main aspects of appendicitis. However, the answer does not provide any specific examples or case studies to support the information, which could have made it more engaging and helpful for the user. Therefore, the answer follows the metric to a good extent.

Step 3: Evaluate the extent to which the m

**Relevance**

The relevance evaluation in this instance evalluated the context and then evaluated the relevance. It mentions that the answer is relevant to the question, but lowers the score as it deemed that the answer is not engaging to the user as there are no specific examples. The Relevance score is 3.5

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [79]:
ground_3, relevance_3, answer_3 = generate_ground_relevance_response(query_3,max_tokens=MAX_TOKENS)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [80]:
print(ground_3)

  Sure! Here's the step-by-step explanation of how I evaluated the answer based on the given criteria:

Step 1: Evaluate if the answer is derived only from the information presented in the context.

The answer provides a list of effective treatments for sudden patchy hair loss, which includes topical corticosteroids, intralesional corticosteroids, oral corticosteroids, topical minoxidil, and immunomodulators. All of these treatments are mentioned in the context as possible options for treating alopecia areata. Therefore, the answer is derived only from the information presented in the context.

Step 2: Evaluate the extent to which the metric is followed.

The answer provides a list of effective treatments for sudden patchy hair loss without mentioning any other causes of hair loss or discussing the potential side effects of these treatments. This is in line with the metric, which requires that the answer should be derived only from the information presented in the context and not inclu

**Groundedness**

The evaluation suffers from some repettition. As it mentions several times "The answer provides a list of effective treatments...". At the end it scores the response with a 5 as the answer is complete and follows the metric to a good extent.

In [81]:
print(relevance_3)

  Sure! Here's the step-by-step explanation of how I would evaluate the context:

Step 1: Identify the main aspects of the question that need to be addressed in order to provide a relevant answer.

Based on the question, the main aspects that need to be addressed are:

* The effective treatments for sudden patchy hair loss (alopecia areata)
* The possible causes of sudden patchy hair loss

Step 2: Evaluate how well the answer addresses each of the main aspects identified in step 1.

The answer provides a list of effective treatments for alopecia areata, including topical corticosteroids, intralesional corticosteroids, oral corticosteroids, topical minoxidil, and immunomodulators. It also mentions the possible causes of sudden patchy hair loss, such as autoimmune disorders, hormonal imbalances, infections, traction alopecia, and genetics.

However, the answer does not provide a comprehensive explanation of each treatment option or their effectiveness, nor does it discuss the potential s

**Relevance**

In evaluating relevance, the model did not assign a numeric score but instead rated the response as “good.” It noted that the answer fell short of being considered “excellent” due to its lack of a comprehensive overview and failure to address all key aspects of the topic. Unlike previous assessments where the model adhered closely to the evaluation protocol, this instance reflects a deviation from that approach, resulting in a qualitative rather than quantitative rating.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [82]:
ground_4, relevance_4, answer_4 = generate_ground_relevance_response(query_4, max_tokens=MAX_TOKENS)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [83]:
print(ground_4)

  Sure, I'd be happy to help you rate the answer!

Based on the input provided, here are the steps that are needed to evaluate the answer as per the metric:

Step 1: Check if the answer is derived only from the information presented in the context.

Yes, the answer is derived only from the information presented in the context. The answer mentions supportive care, prevention of pressure ulcers, rehabilitation therapy, and surgery as recommended treatments for a person who has sustained a physical injury to brain tissue. All of these treatments are mentioned in the context.

Step 2: Evaluate the extent to which the metric is followed.

The answer follows the metric completely. The answer only mentions treatments that are derived from the information presented in the context, and does not mention any treatments that are not supported by the context.

Step 3: Use the previous information to rate the answer using the evaluation criteria.

Based on the previous steps, I would rate the answer

**Groundedness**

In this evaluation the model followed much better the instructions. It scored the answer with a 5 as the answer is derived from the information presented in the context.

In [84]:
print(relevance_4)

  Sure, I can help you with that! To evaluate the context, we need to follow these steps:

Step 1: Identify the main aspects of the question that need to be addressed in the answer.

Based on the question, the main aspects are:

* Treatments for a person who has sustained a physical injury to brain tissue
* The injury resulting in temporary or permanent impairment of brain function

Step 2: Evaluate how well the answer addresses these main aspects.

The answer provides a list of treatments that are recommended for a person who has sustained a physical injury to brain tissue, including supportive care, prevention of pressure ulcers, rehabilitation therapy, surgery, and maintenance of adequate brain perfusion and oxygenation. These treatments address the main aspects of the question directly and comprehensively.

Step 3: Consider whether all and only the important aspects are contained in the answer.

The answer covers all the important aspects of the question, including the different ty

**Relevance**

This relevance evaluation is performed better by the LLM. It follows the steps to evaluate. It asigns a score of 4 and mentions that the answer does not provide additional information. It doesn't mention the expected type of information that is missing. Nevertheless a high score of 4 was given to the answer.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [85]:
ground_5, relevance_5, answer_5 = generate_ground_relevance_response(query_5, max_tokens=MAX_TOKENS)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [86]:
print(ground_5)

  Sure, I can help you rate the answer based on the evaluation criteria you provided. Here's how I would evaluate the answer:

1. The metric is followed completely: The answer provides a comprehensive list of necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and all the steps are derived only from the information presented in the context.
2. Steps to evaluate the answer:
a. Check if the answer is based only on the information presented in the context.
b. Evaluate if the answer provides all the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip.
c. Assess if the answer is comprehensive and covers all aspects of care and recovery.
3. Rating: Based on the evaluation criteria, I would rate the answer as 5 (completely followed). The answer provides a thorough list of precautions and treatment steps for a person who has fractured their leg during a hiking trip, and all the information 

**Groundedness**

The model follows the steps to evaluate and with those considerations it scores the answer with a 5. Afterwards it breaks down the evaluation. Which was a step not needed.

In [87]:
print(relevance_5)

  Sure, I can help you with that! To evaluate the context as per the metric, we need to consider the following steps:

Step 1: Identify the main aspects of the question that need to be addressed in the answer.

Based on the question, the main aspects are:

* Precautions to be taken for a person who has fractured their leg during a hiking trip
* Treatment steps to be followed for the care and recovery of the patient

Step 2: Evaluate whether all and only the important aspects are contained in the answer.

The answer provides a comprehensive list of precautions and treatment steps for a person who has fractured their leg during a hiking trip, covering all the important aspects of the question.

Step 3: Consider whether the answer addresses the main aspects of the question based on the context.

The answer addresses all the main aspects of the question based on the provided context, including rewarming, pain management, immobilization, hygiene, monitoring for infection, and early mobiliza

**Relevance**

In this evaluation, the model adheres more precisely to the given guidelines and assigns a relevance score of 5 to the answer. It concludes that the response effectively addresses the question by offering a thorough and complete set of precautions and treatment measures.

## 11 - Response comparison

In [88]:
# Generate a table to compare all the answers
response_summary = pd.DataFrame(
    {
        "Query": [query_1, query_2, query_3, query_4, query_5],
        "Pure LLM": [response_llm_1, response_llm_2, response_llm_3, response_llm_4, response_llm_5],
        "Prompt Engineering": [response_eng_1, response_eng_2, response_eng_3, response_eng_4, response_eng_5],
        "RAG": [response_rag_1, response_rag_2, response_rag_3, response_rag_4, response_rag_5],
        "Fine-tuning": [response_fine_1, response_fine_2, response_fine_3, response_fine_4, response_fine_5],
        "Answer": [answer_1, answer_2, answer_3, answer_4, answer_5],
        "Ground Truth": [ground_1, ground_2, ground_3, ground_4, ground_5],
        "Relevance": [relevance_1, relevance_2, relevance_3, relevance_4, relevance_5],
    }
)

In [89]:
# Print the response summary
response_summary

,Query,Pure LLM,Prompt Engineering,RAG,Fine-tuning,Answer,Ground Truth,Relevance
0,\nWhat is the protocol for managing sepsis in a critical care unit?\n,"\nSepsis is a life-threatening condition that can arise from an infection, and it is important to have a clear protocol for managing it in a critical care unit. Here are some key components of a sepsis management protocol:\n\n1. Early recognition and identification: The first step in managing sepsis is to recognize the signs and symptoms early on, such as fever, tachycardia, tachypnea, and confusion. The patient's medical history, including any recent infections or surgeries, should also be reviewed.\n2. Rapid laboratory testing: Blood cultures and other laboratory tests, such as complete blood counts and serum lactate levels, should be performed promptly to confirm the presence of an infection and assess its severity.\n3. Administration of antibiotics: Broad-spectrum antibiotics should be administered as soon as possible, ideally within the first hour of recognition of sepsis. The choice of antibiotics should be guided by the suspected source of the infection and the patient's allergies and medical history.\n4. Fluid resuscitation: Sepsis can lead to hypovolemia and hypotension, so fluid resuscitation is critical to maintain blood pressure and perfusion of vital organs. Crystalloid fluids such as lactated Ringer's solution or normal saline should be administered initially, and vasopressors may be added if necessary.\n5. Monitoring and assessment: Patients with sepsis should be closely monitored for signs of organ dysfunction, such as altered mental status, respiratory failure, cardiac dysfunction, and renal failure. These signs should be documented and reported to the critical care team.\n6. Management of underlying infections: The source of the infection should be identified and treated appropriately, such as with antibiotics for bacterial infections or antifungals for fungal infections.\n7. Supportive care: Patients with sepsis may require supportive care, such as mechanical ventilation, dialysis, or cardiac support, depending on the severity of their illness.\n8. Protocol-driven management: A sepsis management protocol should be followed to ensure that all patients receive consistent and evidence-based care. This can include protocols for fluid resuscitation, antibiotic administration, and monitoring for organ dysfunction.\n9. Multidisciplinary team approach: Sepsis management should involve a multidisciplinary team of healthcare professionals, including intensivists, nurses, respiratory therapists, and pharmacists. This team should work together to provide coordinated care and monitor the patient's progress.\n10. Continuous quality improvement: The sepsis management protocol should be regularly reviewed and updated to ensure that it is evidence-based and effective in improving patient outcomes. This can involve monitoring key performance indicators, such as mortality rates and length of stay, and making adjustments to the protocol as needed.\n\nBy following a structured sepsis management protocol, critical care units can provide high-quality care to patients with sepsis and improve their chances of survival and recovery.","Protocol for Managing Sepsis in the Critical Care Unit A standardized, evidence-based sepsis protocol is paramount... Sepcertain situations as gupta Proceedure and/Brown University Cente for biovettual Inameduares). C. Empiric Antimucobacterial Medication Driubbed throughout these procedures arto bemonitorvital sifns of severeklife-threeting 15(-points;6r hemodiynamicsand respidony ptotected sepsi;2d/1 reapiratorsupport if ptimateindicat ed. IV Fluid Stratum Bund Blood;NS), along s fluid s overevance or dobutexorshop erive shock refrotocol should aim to ideal Patient Comfurter(if necessary), mane possible airwa Ed; an eshetic ed with siderable clinlicallcnicsoope of stus, pffering mocules ptoclract a clinigc dclepmension: D. Pain \nControl, Restraint s

## Actionable Insights and Business Recommendations

**Actionable Insights**

When comparing performance across methodologies, each approach demonstrates distinct strengths. Pure LLMs or vanilla LLMs tend to produce comprehensive and detailed responses, though they may occasionally lack specificity or direct relevance to the context in some cases. Prompt engineering enhances structure and clarity, often guiding the model through step-by-step reasoning to improve relevance and accuracy. Retrieval-Augmented Generation (RAG) stands out by incorporating external sources, yielding contextually grounded and complete responses.

When evaluating accuracy and relevance across methodologies, RAG consistently performed better due to their grounding in domain-specific context or targeted training data. These approaches produced responses that were more aligned with the question and current medical or technical standards. In contrast, Pure LLM and Prompt Engineering methods, while still informative and well-structured, occasionally generated outputs that lacked contextual specificity or leaned on generalized knowledge. This distinction underscores the value of context-aware generation for high-precision tasks.

Responses for sepsis management, appendicitis, and brain injury treatments were highly detailed and actionable across all methodologies.For hair loss and leg fractures, RAG provided more practical and context-aware recommendations.

The Groundedness and Relevance evaluators  favors format over fidelity. Scoring that rewards context-only derivation and completeness. These evaluators don't score if the information is up to date to current medical standards.

**Business Recomendations**

1. Selecting the appropriate LLM strategy is crucial for the intended audience.
- For healthcare professionals, the priority lies in receiving highly specialized and clinically precise responses. This makes Retrieval-Augmented Generation (RAG) the most suitable methodology, as it excels in delivering timely, context-rich information—ideal for complex cases like sepsis protocols or fracture care pathways.
- In contrast, Prompt Engineering serves well in general education and non-critical contexts, offering accessible and well-organized explanations tailored to lay users who seek a broad understanding of medical procedures without requiring deep clinical detail.

2. Strengthen evaluators: The Groundedness and Relevance evaluations have to ensure that the model answers  adhere to clinical standards, reflect up-to-date evidence, and deliver precise, context-specific guidance without unnecessary detail.

3. The responses have to prioritize clarity, engagement, and actionable value for the healthcare professional.

This analysis highlights that RAG is superior for medical applications, offering context-aware, accurate, and actionable insights. Businesses should invest in these methodologies while refining evaluation frameworks and customization for end-users. This approach ensures reliable, scalable, and compliant AI solutions in healthcare.

<font size=6 color='blue'>Power Ahead</font>
___